<a href="https://colab.research.google.com/github/yhshengjy/ClinPKPD/blob/main/Notebook3_renal_function_dose_adjustment_english.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3: Changes in Renal Function and Dose Adjustment

This notebook is the third module of the Clinical Pharmacy PK/PD Interactive Simulation Platform.

In the first two notebooks, we learned about:

- Changes in plasma drug concentration after a single dose
- One-compartment models for oral and intravenous administration
- Multiple dosing and steady-state concentration
- The therapeutic window and a basic Emax pharmacodynamic model

This section further connects PK/PD models with patient characteristics, focusing on:

> How does drug clearance change when renal function declines?  
> After clearance changes, how do plasma concentration, AUC, half-life, and toxicity risk change?  
> How can we use model-based thinking to understand dose adjustment?

The core logic of this notebook is:

$$
Patient\ renal\ function \rightarrow Drug\ clearance \rightarrow Concentration(t) \rightarrow Exposure \rightarrow Dose\ adjustment
$$

Please note: This notebook is intended for educational simulation only and does not replace real clinical prescribing. Dose adjustment in real patients must consider the prescribing information, institutional guidelines, renal replacement therapy modality, infection severity, TDM results, and clinical response.


## 1. Learning Objectives

After completing this notebook, you should be able to:

1. Explain why renal function affects drug clearance and systemic exposure.
2. Use the Cockcroft-Gault equation to estimate creatinine clearance (CrCl).
3. Distinguish among CrCl, eGFR, and their roles in drug dose adjustment.
4. Describe how reduced renal function changes AUC, half-life, Cmax, Cmin, and accumulation risk.
5. Compare common renal dose adjustment strategies, including dose reduction and interval extension.

## 2. Clinical Scenario

A patient needs an antibacterial drug that is primarily cleared by the kidneys for the treatment of an infection.

The standard dosing regimen is:

$$
500\ mg\ q24h
$$

However, the patient is older, has an elevated serum creatinine, and has reduced estimated creatinine clearance.

At this point, the clinical pharmacist needs to consider:

- What is the patient's approximate renal function?
- Will drug clearance decrease?
- If the standard dose is still used, will accumulation occur?
- Should each dose be reduced, or should the dosing interval be extended?
- How can the concentration-time curve help determine whether an adjusted regimen is more reasonable?

This notebook will use interactive simulations to answer these questions step by step.


## 3. Renal Function and Drug Clearance

Total drug clearance can be simplified as:

$$
CL_{total} = CL_{renal} + CL_{nonrenal}
$$

where:

| Symbol | Meaning |
|---|---|
| $CL_{renal}$ | Renal clearance |
| $CL_{nonrenal}$ | Nonrenal clearance, such as hepatic metabolism, biliary excretion, and other pathways |
| $CL_{total}$ | Total clearance |

For drugs that are primarily cleared by the kidneys, a decline in renal function can substantially reduce total clearance.  
For drugs that are primarily metabolized by the liver, a decline in renal function may have a smaller effect on total clearance.

For this educational simulation, we use the fraction excreted unchanged in urine, $f_e$, to represent the renal-dependent component of drug clearance:

$$
f_e = \frac{CL_{renal}}{CL_{total}}
$$

A larger $f_e$ means the drug depends more heavily on renal clearance, so a decline in renal function has a more obvious effect on drug exposure.


## 4. Estimating Creatinine Clearance (CrCl)

In clinical practice, the Cockcroft-Gault equation is commonly used to estimate creatinine clearance:

$$
CrCl = \frac{(140 - Age) \times Weight}{72 \times SCr}
$$

For female patients, the result is commonly multiplied by 0.85:

$$
CrCl_{female} = CrCl \times 0.85
$$

where:

| Symbol | Meaning | Unit |
|---|---|---|
| Age | Age | years |
| Weight | Body weight | kg |
| SCr | Serum creatinine | mg/dL |
| CrCl | Creatinine clearance | mL/min |

If serum creatinine is reported in μmol/L, it can be approximately converted as:

$$
SCr(mg/dL) = \frac{SCr(\mu mol/L)}{88.4}
$$

Note: The Cockcroft-Gault equation has limitations, especially in patients with extreme body weight, acute kidney injury, abnormal muscle mass, frailty in older adults, pregnancy, and other special conditions. In real clinical practice, interpretation should be combined with patient-specific factors and institutional standards.


## 5. CrCl, eGFR, and Dose Adjustment

Clinical laboratory reports often provide eGFR, usually in the unit:

$$
mL/min/1.73m^2
$$

However, many drug prescribing information documents provide dose adjustment recommendations based on CrCl, usually in the unit:

$$
mL/min
$$

Therefore, when adjusting drug doses, pay attention to:

- Whether the prescribing information uses CrCl or eGFR.
- Whether the equation is standardized to body surface area.
- Whether the patient has acute kidney injury or rapidly changing renal function.
- Whether the patient is receiving hemodialysis, peritoneal dialysis, or CRRT.

This notebook uses the Cockcroft-Gault equation to estimate CrCl mainly for educational simulation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


def scr_umol_l_to_mg_dl(scr_umol_l):
    """
    Convert serum creatinine from micromol/L to mg/dL.
    """
    return scr_umol_l / 88.4


def height_cm_to_inches(height_cm):
    """
    Convert height from cm to inches.
    """
    return height_cm / 2.54


def calculate_ibw(sex, height_cm):
    """
    Calculate ideal body weight using Devine-style equations.
    This is used here only for teaching simulation.
    """
    height_in = height_cm_to_inches(height_cm)
    inches_over_5ft = max(height_in - 60, 0)

    if sex == "Male":
        ibw = 50 + 2.3 * inches_over_5ft
    else:
        ibw = 45.5 + 2.3 * inches_over_5ft

    return ibw


def choose_weight_for_cg(sex, height_cm, actual_weight_kg, method):
    """
    Choose body weight for Cockcroft-Gault calculation.
    """
    ibw = calculate_ibw(sex, height_cm)
    adjusted_bw = ibw + 0.4 * (actual_weight_kg - ibw)

    if method == "Actual body weight":
        selected_weight = actual_weight_kg
    elif method == "Ideal body weight":
        selected_weight = ibw
    else:
        selected_weight = adjusted_bw

    return selected_weight, ibw, adjusted_bw


def cockcroft_gault_crcl(age_years, sex, weight_kg, scr_mg_dl):
    """
    Estimate creatinine clearance using Cockcroft-Gault equation.
    """
    crcl = ((140 - age_years) * weight_kg) / (72 * scr_mg_dl)

    if sex == "Female":
        crcl *= 0.85

    return crcl


def classify_crcl(crcl_ml_min):
    """
    Simple renal function category for medication dosing discussion.
    """
    if crcl_ml_min >= 90:
        return "Normal or near normal"
    elif crcl_ml_min >= 60:
        return "Mild decrease"
    elif crcl_ml_min >= 30:
        return "Moderate decrease"
    elif crcl_ml_min >= 15:
        return "Severe decrease"
    else:
        return "Kidney failure range"

## 6. Interactive Simulation 1: Estimating Patient CrCl

The following simulation estimates CrCl using the patient's age, sex, body weight, height, and serum creatinine.

You can choose the serum creatinine unit:

- mg/dL
- μmol/L

You can also choose the body weight used in the Cockcroft-Gault equation:

- Actual body weight
- Ideal body weight
- Adjusted body weight

Please observe:

- How does CrCl change as age increases?
- How does CrCl change as SCr increases?
- Under the same SCr, how do sex and body weight affect CrCl?
- Does the choice of body weight method substantially change CrCl?


In [ ]:
def plot_crcl_calculator(
    age_years=70,
    sex="Male",
    actual_weight_kg=70,
    height_cm=170,
    scr_value=1.5,
    scr_unit="mg/dL",
    weight_method="Actual body weight"
):
    if scr_unit == "mg/dL":
        scr_mg_dl = scr_value
    else:
        scr_mg_dl = scr_umol_l_to_mg_dl(scr_value)

    selected_weight, ibw, adjusted_bw = choose_weight_for_cg(
        sex=sex,
        height_cm=height_cm,
        actual_weight_kg=actual_weight_kg,
        method=weight_method
    )

    crcl = cockcroft_gault_crcl(
        age_years=age_years,
        sex=sex,
        weight_kg=selected_weight,
        scr_mg_dl=scr_mg_dl
    )

    category = classify_crcl(crcl)

    summary = pd.DataFrame({
        "Parameter": [
            "Age",
            "Sex",
            "Height",
            "Actual body weight",
            "Ideal body weight",
            "Adjusted body weight",
            "Selected weight method",
            "Selected weight",
            "SCr",
            "Estimated CrCl",
            "Renal function category"
        ],
        "Value": [
            f"{age_years:.0f} years",
            sex,
            f"{height_cm:.1f} cm",
            f"{actual_weight_kg:.1f} kg",
            f"{ibw:.1f} kg",
            f"{adjusted_bw:.1f} kg",
            weight_method,
            f"{selected_weight:.1f} kg",
            f"{scr_mg_dl:.2f} mg/dL",
            f"{crcl:.1f} mL/min",
            category
        ]
    })

    display(summary)


interact(
    plot_crcl_calculator,
    age_years=IntSlider(value=70, min=18, max=95, step=1, description="Age"),
    sex=Dropdown(options=["Male", "Female"], value="Male", description="Sex"),
    actual_weight_kg=FloatSlider(value=70, min=35, max=150, step=1, description="Weight"),
    height_cm=FloatSlider(value=170, min=140, max=200, step=1, description="Height"),
    scr_value=FloatSlider(value=1.5, min=0.4, max=8.0, step=0.1, description="SCr"),
    scr_unit=Dropdown(options=["mg/dL", "umol/L"], value="mg/dL", description="SCr unit"),
    weight_method=Dropdown(
        options=["Actual body weight", "Ideal body weight", "Adjusted body weight"],
        value="Actual body weight",
        description="Weight"
    )
);

interactive(children=(IntSlider(value=70, description='Age', max=95, min=18), Dropdown(description='Sex', opti…

## 7. Observation Task 1: Factors Affecting CrCl

Please complete the following tasks:

### Task A: Standard Patient

Set:

- Age = 70 years
- Sex = Male
- Weight = 70 kg
- Height = 170 cm
- SCr = 1.5 mg/dL
- Weight method = Actual body weight

Record:

- Estimated CrCl
- Renal function category

### Task B: Change Age Only

Change Age to 40 years.

Observe:

- Does CrCl increase?
- Why does age affect CrCl under the same SCr?

### Task C: Change SCr Only

Change Age back to 70 years, then change SCr to 3.0 mg/dL.

Observe:

- Does CrCl decrease?
- Does the renal function category change?

### Task D: Change the Body Weight Method

Keep the other parameters unchanged and select each of the following:

- Actual body weight
- Ideal body weight
- Adjusted body weight

Observe:

- Does CrCl change?
- What does this imply about the impact of body weight selection during dose adjustment?


## 8. From Renal Function to Drug Clearance

To connect patient renal function with drug clearance, this notebook uses a simplified model:

$$
CL_{patient} = CL_{normal} \times \left[(1-f_e) + f_e \times \frac{CrCl_{patient}}{CrCl_{normal}}\right]
$$

where:

| Symbol | Meaning |
|---|---|
| $CL_{patient}$ | The patient's current total clearance |
| $CL_{normal}$ | Total clearance in a patient with normal renal function |
| $f_e$ | Fraction excreted unchanged in urine |
| $CrCl_{patient}$ | The patient's estimated creatinine clearance |
| $CrCl_{normal}$ | Reference normal creatinine clearance, set to 100 mL/min in this section |

When $f_e = 1$, the drug is completely dependent on renal clearance.  
When $f_e = 0$, drug clearance is essentially independent of renal function.

Therefore, a decline in renal function affects different drugs to different degrees.


In [ ]:
def estimate_patient_clearance(cl_normal_l_h, crcl_patient_ml_min, fe_renal, crcl_normal_ml_min=100):
    """
    Estimate patient clearance based on renal function and fraction excreted renally.
    """
    renal_function_ratio = crcl_patient_ml_min / crcl_normal_ml_min
    clearance_ratio = (1 - fe_renal) + fe_renal * renal_function_ratio
    clearance_ratio = max(clearance_ratio, 0.02)
    cl_patient_l_h = cl_normal_l_h * clearance_ratio

    return cl_patient_l_h, clearance_ratio


def one_compartment_iv_bolus(t, dose_mg, vd_l, cl_l_h):
    """
    One-compartment IV bolus model.
    """
    k_elim = cl_l_h / vd_l
    concentration = (dose_mg / vd_l) * np.exp(-k_elim * t)
    auc = dose_mg / cl_l_h
    half_life = np.log(2) / k_elim

    return concentration, k_elim, half_life, auc


def one_compartment_oral(t, dose_mg, vd_l, cl_l_h, ka_h, bioavailability):
    """
    One-compartment oral model with first-order absorption.
    """
    k_elim = cl_l_h / vd_l

    if np.isclose(ka_h, k_elim):
        concentration = bioavailability * dose_mg / vd_l * k_elim * t * np.exp(-k_elim * t)
    else:
        concentration = (
            bioavailability * dose_mg * ka_h / (vd_l * (ka_h - k_elim))
            * (np.exp(-k_elim * t) - np.exp(-ka_h * t))
        )

    concentration = np.maximum(concentration, 0)
    auc = bioavailability * dose_mg / cl_l_h
    half_life = np.log(2) / k_elim

    return concentration, k_elim, half_life, auc


def calculate_basic_pk_metrics(t, concentration, auc, half_life):
    """
    Calculate basic PK metrics.
    """
    cmax = np.max(concentration)
    tmax = t[np.argmax(concentration)]

    return {
        "Cmax": cmax,
        "Tmax": tmax,
        "AUC": auc,
        "Half-life": half_life
    }

## 9. Interactive Simulation 2: How Does Reduced Renal Function Change the Single-Dose Curve?

The following simulation shows concentration changes after a dose in patients with different levels of renal function.

You can adjust:

- CrCl: patient creatinine clearance
- $f_e$: fraction of drug clearance that is renal
- CL normal: clearance under normal renal function
- Vd: volume of distribution
- Route: intravenous or oral administration

Please observe:

- When CrCl decreases, does patient CL decrease?
- Does the half-life become longer?
- Does AUC increase?
- When $f_e$ is larger, is the impact of reduced renal function more obvious?


In [ ]:
def plot_renal_function_single_dose(
    route="Oral",
    dose_mg=500,
    vd_l=70,
    cl_normal_l_h=8,
    crcl_patient_ml_min=40,
    fe_renal=0.8,
    ka_h=1.2,
    bioavailability=0.9,
    t_end_h=72
):
    t = np.linspace(0, t_end_h, 1000)

    cl_patient_l_h, clearance_ratio = estimate_patient_clearance(
        cl_normal_l_h=cl_normal_l_h,
        crcl_patient_ml_min=crcl_patient_ml_min,
        fe_renal=fe_renal
    )

    if route == "IV bolus":
        conc_normal, _, half_life_normal, auc_normal = one_compartment_iv_bolus(
            t=t, dose_mg=dose_mg, vd_l=vd_l, cl_l_h=cl_normal_l_h
        )
        conc_patient, _, half_life_patient, auc_patient = one_compartment_iv_bolus(
            t=t, dose_mg=dose_mg, vd_l=vd_l, cl_l_h=cl_patient_l_h
        )
    else:
        conc_normal, _, half_life_normal, auc_normal = one_compartment_oral(
            t=t, dose_mg=dose_mg, vd_l=vd_l, cl_l_h=cl_normal_l_h,
            ka_h=ka_h, bioavailability=bioavailability
        )
        conc_patient, _, half_life_patient, auc_patient = one_compartment_oral(
            t=t, dose_mg=dose_mg, vd_l=vd_l, cl_l_h=cl_patient_l_h,
            ka_h=ka_h, bioavailability=bioavailability
        )

    metrics_normal = calculate_basic_pk_metrics(t, conc_normal, auc_normal, half_life_normal)
    metrics_patient = calculate_basic_pk_metrics(t, conc_patient, auc_patient, half_life_patient)

    fig, ax = plt.subplots()
    ax.plot(t, conc_normal, linewidth=2, label="Normal renal function")
    ax.plot(t, conc_patient, linewidth=2, linestyle="--", label="Patient renal function")

    ax.set_title("Effect of Renal Function on Single-Dose PK")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Route",
            "Patient CrCl",
            "Renal fraction fe",
            "Normal CL",
            "Patient CL",
            "Clearance ratio",
            "Normal half-life",
            "Patient half-life",
            "Normal AUC",
            "Patient AUC",
            "AUC fold-change"
        ],
        "Value": [
            route,
            f"{crcl_patient_ml_min:.1f} mL/min",
            f"{fe_renal:.2f}",
            f"{cl_normal_l_h:.2f} L/h",
            f"{cl_patient_l_h:.2f} L/h",
            f"{clearance_ratio:.2f}",
            f"{metrics_normal['Half-life']:.2f} h",
            f"{metrics_patient['Half-life']:.2f} h",
            f"{metrics_normal['AUC']:.2f} mg*h/L",
            f"{metrics_patient['AUC']:.2f} mg*h/L",
            f"{metrics_patient['AUC'] / metrics_normal['AUC']:.2f}"
        ]
    })

    display(summary)


interact(
    plot_renal_function_single_dose,
    route=Dropdown(options=["IV bolus", "Oral"], value="Oral", description="Route"),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=70, min=10, max=150, step=5, description="Vd"),
    cl_normal_l_h=FloatSlider(value=8, min=1, max=20, step=0.5, description="CL normal"),
    crcl_patient_ml_min=FloatSlider(value=40, min=5, max=120, step=5, description="CrCl"),
    fe_renal=FloatSlider(value=0.8, min=0, max=1, step=0.05, description="fe"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="F"),
    t_end_h=FloatSlider(value=72, min=24, max=168, step=12, description="Time")
);

## 10. Observation Task 2: Renal Function, $f_e$, and Changes in Exposure

Please complete the following tasks:

### Task A: Moderate Decline in Renal Function

Set:

- Route = Oral
- Dose = 500 mg
- Vd = 70 L
- CL normal = 8 L/h
- CrCl = 40 mL/min
- $f_e$ = 0.8
- F = 0.9

Record:

- Patient CL
- Patient half-life
- Patient AUC
- AUC fold-change

### Task B: Further Decline in Renal Function

Change CrCl to 15 mL/min.

Observe:

- Does patient CL decrease further?
- Does the AUC fold-change increase?
- Does the half-life become longer?

### Task C: Change the Fraction of Renal Clearance

Keep CrCl = 15 mL/min and set:

- $f_e$ = 0.2
- $f_e$ = 0.8
- $f_e$ = 1.0

Observe:

- In which case does AUC increase the most?
- Why does this explain that not all drugs require the same degree of renal dose adjustment?


## 11. Basic Principles of Dose Adjustment

After renal function declines, drug clearance decreases. To avoid excessive exposure, common dose adjustment strategies include:

### Strategy 1: Reduce Each Dose While Keeping the Dosing Interval Unchanged

For example:

$$
500\ mg\ q24h \rightarrow 250\ mg\ q24h
$$

This approach can reduce exposure while maintaining a relatively stable dosing rhythm.

### Strategy 2: Keep Each Dose Unchanged and Extend the Dosing Interval

For example:

$$
500\ mg\ q24h \rightarrow 500\ mg\ q48h
$$

This approach can preserve a relatively high peak concentration, but the trough concentration within the dosing interval may be lower.

### Strategy 3: Reduce the Dose and Extend the Interval at the Same Time

For example:

$$
500\ mg\ q24h \rightarrow 250\ mg\ q48h
$$

This approach is often used for severe renal impairment or drugs with a narrow therapeutic window.

The key to dose adjustment is not mechanically applying a formula, but understanding that:

$$
Maintenance\ dose\ rate \propto CL
$$

In other words, the maintenance dose rate should usually decrease as clearance decreases.


In [ ]:
def concentration_after_dose(t_after_dose, dose_mg, vd_l, cl_l_h, route, ka_h, bioavailability):
    """
    Concentration contribution after a single dose.
    """
    t_after_dose = np.asarray(t_after_dose)
    concentration = np.zeros_like(t_after_dose, dtype=float)
    mask = t_after_dose >= 0
    dt = t_after_dose[mask]

    k_elim = cl_l_h / vd_l

    if route == "IV bolus":
        concentration[mask] = (dose_mg / vd_l) * np.exp(-k_elim * dt)
    else:
        if np.isclose(ka_h, k_elim):
            concentration[mask] = bioavailability * dose_mg / vd_l * k_elim * dt * np.exp(-k_elim * dt)
        else:
            concentration[mask] = (
                bioavailability * dose_mg * ka_h / (vd_l * (ka_h - k_elim))
                * (np.exp(-k_elim * dt) - np.exp(-ka_h * dt))
            )

    return np.maximum(concentration, 0)


def multiple_dose_concentration(t, dose_mg, tau_h, duration_h, vd_l, cl_l_h,
                                route="Oral", ka_h=1.2, bioavailability=0.9,
                                first_dose_mg=None):
    """
    Multiple-dose concentration profile with optional first dose.
    """
    concentration = np.zeros_like(t, dtype=float)
    dose_times = np.arange(0, duration_h + 1e-9, tau_h)

    for i, dose_time in enumerate(dose_times):
        current_dose = dose_mg
        if i == 0 and first_dose_mg is not None:
            current_dose = first_dose_mg

        concentration += concentration_after_dose(
            t_after_dose=t - dose_time,
            dose_mg=current_dose,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            route=route,
            ka_h=ka_h,
            bioavailability=bioavailability
        )

    return concentration, dose_times


def steady_state_interval_concentration(
    dose_mg, tau_h, vd_l, cl_l_h, route="Oral",
    ka_h=1.2, bioavailability=0.9, n_points=4001
):
    """
    Steady-state concentration profile over one complete dosing interval.

    The profile is calculated analytically for a one-compartment model with
    first-order elimination and, for oral dosing, first-order absorption.
    """
    t_interval = np.linspace(0, tau_h, n_points)
    k_elim = cl_l_h / vd_l

    if route == "IV bolus":
        denominator = 1 - np.exp(-k_elim * tau_h)
        concentration = (
            dose_mg / vd_l
            * np.exp(-k_elim * t_interval)
            / denominator
        )
    else:
        if np.isclose(ka_h, k_elim):
            r = np.exp(-k_elim * tau_h)
            concentration = (
                bioavailability * dose_mg / vd_l
                * k_elim * np.exp(-k_elim * t_interval)
                * (
                    t_interval / (1 - r)
                    + tau_h * r / (1 - r) ** 2
                )
            )
        else:
            concentration = (
                bioavailability * dose_mg * ka_h / (vd_l * (ka_h - k_elim))
                * (
                    np.exp(-k_elim * t_interval) / (1 - np.exp(-k_elim * tau_h))
                    - np.exp(-ka_h * t_interval) / (1 - np.exp(-ka_h * tau_h))
                )
            )

    return t_interval, np.maximum(concentration, 0)


def steady_state_interval_metrics(
    dose_mg, tau_h, vd_l, cl_l_h, mec, mtc,
    route="Oral", ka_h=1.2, bioavailability=0.9
):
    """
    Calculate steady-state exposure metrics over one complete dosing interval.
    """
    t_interval, concentration = steady_state_interval_concentration(
        dose_mg=dose_mg,
        tau_h=tau_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    cmax_ss = np.max(concentration)
    cmin_ss = np.min(concentration)

    # For linear PK, steady-state AUC over one dosing interval is Dose/CL
    # (multiplied by F for oral administration).
    f_auc = 1.0 if route == "IV bolus" else bioavailability
    auc_tau_ss = f_auc * dose_mg / cl_l_h
    cavg_ss = auc_tau_ss / tau_h

    pct_below_mec = (
        np.trapz((concentration < mec).astype(float), t_interval) / tau_h * 100
    )
    pct_above_mtc = (
        np.trapz((concentration > mtc).astype(float), t_interval) / tau_h * 100
    )

    return {
        "Cmax_ss": cmax_ss,
        "Cmin_ss": cmin_ss,
        "AUC_tau_ss": auc_tau_ss,
        "Cavg_ss": cavg_ss,
        "Pct_time_below_MEC": pct_below_mec,
        "Pct_time_above_MTC": pct_above_mtc
    }


## 12. Interactive Simulation 3: Comparing Different Dose Adjustment Strategies

The simulation compares four conditions:

1. **Normal reference**: the standard regimen under normal renal function
2. **No adjustment**: the original dose and dosing interval are maintained despite reduced renal function
3. **Reduce dose**: each dose is reduced in proportion to the decline in clearance while the dosing interval is unchanged
4. **Extend interval**: each dose is maintained while the dosing interval is extended in proportion to the decline in clearance

The summary table uses metrics calculated over **one complete dosing interval at steady state**.

Focus on the following:

- **Steady-state Cmax**: how does peak concentration change with each adjustment strategy?
- **Steady-state Cmin**: does interval extension produce a lower trough concentration?
- **AUCτ,ss**: what is the total exposure over one complete dosing interval? Because AUCτ spans different lengths of time when τ differs, it should not be used alone to compare average exposure across regimens with different dosing intervals.
- **Cavg,ss**: use this metric to compare average steady-state exposure across regimens with different dosing intervals and with the normal reference.
- **Time above MTC / below MEC (% of interval)**: what proportion of a complete dosing interval is above or below the illustrative concentration thresholds?

> **Note:** MEC and MTC in this notebook are **illustrative educational thresholds** used to demonstrate concentration–time concepts and regimen differences. They do not represent the therapeutic range or prescribing recommendations for a specific drug.


In [ ]:
def plot_dose_adjustment_strategies(
    route="Oral",
    usual_dose_mg=500,
    usual_tau_h=24,
    vd_l=70,
    cl_normal_l_h=8,
    crcl_patient_ml_min=30,
    fe_renal=0.8,
    ka_h=1.2,
    bioavailability=0.9,
    mec=0.5,
    mtc=8,
    duration_days=7
):
    duration_h = duration_days * 24
    t = np.linspace(0, duration_h, 2500)

    cl_patient_l_h, clearance_ratio = estimate_patient_clearance(
        cl_normal_l_h=cl_normal_l_h,
        crcl_patient_ml_min=crcl_patient_ml_min,
        fe_renal=fe_renal
    )

    reduced_dose_mg = usual_dose_mg * clearance_ratio
    extended_tau_h = usual_tau_h / clearance_ratio
    extended_tau_h = min(max(extended_tau_h, usual_tau_h), 120)

    # Normal renal-function reference regimen
    conc_normal_reference, _ = multiple_dose_concentration(
        t=t, dose_mg=usual_dose_mg, tau_h=usual_tau_h, duration_h=duration_h,
        vd_l=vd_l, cl_l_h=cl_normal_l_h, route=route, ka_h=ka_h,
        bioavailability=bioavailability
    )

    conc_standard, _ = multiple_dose_concentration(
        t=t, dose_mg=usual_dose_mg, tau_h=usual_tau_h, duration_h=duration_h,
        vd_l=vd_l, cl_l_h=cl_patient_l_h, route=route, ka_h=ka_h,
        bioavailability=bioavailability
    )

    conc_reduced_dose, _ = multiple_dose_concentration(
        t=t, dose_mg=reduced_dose_mg, tau_h=usual_tau_h, duration_h=duration_h,
        vd_l=vd_l, cl_l_h=cl_patient_l_h, route=route, ka_h=ka_h,
        bioavailability=bioavailability
    )

    conc_extended_interval, _ = multiple_dose_concentration(
        t=t, dose_mg=usual_dose_mg, tau_h=extended_tau_h, duration_h=duration_h,
        vd_l=vd_l, cl_l_h=cl_patient_l_h, route=route, ka_h=ka_h,
        bioavailability=bioavailability
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(t, conc_normal_reference, linewidth=2, linestyle="-.", label="Normal reference")
    ax.plot(t, conc_standard, linewidth=2, label="No adjustment")
    ax.plot(t, conc_reduced_dose, linewidth=2, linestyle="--", label="Reduce dose")
    ax.plot(t, conc_extended_interval, linewidth=2, linestyle=":", label="Extend interval")
    ax.axhline(mec, linestyle="--", label=f"MEC = {mec:.1f} mg/L")
    ax.axhline(mtc, linestyle="--", label=f"MTC = {mtc:.1f} mg/L")
    ax.fill_between(t, mec, mtc, alpha=0.12, label="Illustrative therapeutic window")

    ax.set_title("Dose Adjustment Strategies in Renal Impairment")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    metrics_normal = steady_state_interval_metrics(
        dose_mg=usual_dose_mg, tau_h=usual_tau_h, vd_l=vd_l,
        cl_l_h=cl_normal_l_h, mec=mec, mtc=mtc, route=route,
        ka_h=ka_h, bioavailability=bioavailability
    )
    metrics_standard = steady_state_interval_metrics(
        dose_mg=usual_dose_mg, tau_h=usual_tau_h, vd_l=vd_l,
        cl_l_h=cl_patient_l_h, mec=mec, mtc=mtc, route=route,
        ka_h=ka_h, bioavailability=bioavailability
    )
    metrics_reduced = steady_state_interval_metrics(
        dose_mg=reduced_dose_mg, tau_h=usual_tau_h, vd_l=vd_l,
        cl_l_h=cl_patient_l_h, mec=mec, mtc=mtc, route=route,
        ka_h=ka_h, bioavailability=bioavailability
    )
    metrics_extended = steady_state_interval_metrics(
        dose_mg=usual_dose_mg, tau_h=extended_tau_h, vd_l=vd_l,
        cl_l_h=cl_patient_l_h, mec=mec, mtc=mtc, route=route,
        ka_h=ka_h, bioavailability=bioavailability
    )

    metric_names = [
        "Dose",
        "Interval",
        "Dose rate",
        "Steady-state Cmax",
        "Steady-state Cmin",
        "AUCτ,ss (per dosing interval)",
        "Cavg,ss",
        "Time below MEC (% of interval)",
        "Time above MTC (% of interval)"
    ]

    def format_column(dose, tau, metrics):
        return [
            f"{dose:.0f} mg",
            f"{tau:.1f} h",
            f"{dose / tau:.1f} mg/h",
            f"{metrics['Cmax_ss']:.2f} mg/L",
            f"{metrics['Cmin_ss']:.2f} mg/L",
            f"{metrics['AUC_tau_ss']:.2f} mg·h/L",
            f"{metrics['Cavg_ss']:.2f} mg/L",
            f"{metrics['Pct_time_below_MEC']:.1f}%",
            f"{metrics['Pct_time_above_MTC']:.1f}%"
        ]

    summary = pd.DataFrame({
        "Metric": metric_names,
        "Normal reference": format_column(usual_dose_mg, usual_tau_h, metrics_normal),
        "No adjustment": format_column(usual_dose_mg, usual_tau_h, metrics_standard),
        "Reduce dose": format_column(reduced_dose_mg, usual_tau_h, metrics_reduced),
        "Extend interval": format_column(usual_dose_mg, extended_tau_h, metrics_extended)
    })

    display(summary)

    info = pd.DataFrame({
        "Parameter": [
            "Patient CrCl",
            "Normal CL",
            "Patient CL",
            "Clearance ratio",
            "Renal fraction fe"
        ],
        "Value": [
            f"{crcl_patient_ml_min:.1f} mL/min",
            f"{cl_normal_l_h:.2f} L/h",
            f"{cl_patient_l_h:.2f} L/h",
            f"{clearance_ratio:.2f}",
            f"{fe_renal:.2f}"
        ]
    })

    display(info)


interact(
    plot_dose_adjustment_strategies,
    route=Dropdown(options=["IV bolus", "Oral"], value="Oral", description="Route"),
    usual_dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    usual_tau_h=FloatSlider(value=24, min=6, max=48, step=6, description="Tau"),
    vd_l=FloatSlider(value=70, min=10, max=150, step=5, description="Vd"),
    cl_normal_l_h=FloatSlider(value=8, min=1, max=20, step=0.5, description="CL normal"),
    crcl_patient_ml_min=FloatSlider(value=30, min=5, max=120, step=5, description="CrCl"),
    fe_renal=FloatSlider(value=0.8, min=0, max=1, step=0.05, description="fe"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="F"),
    mec=FloatSlider(value=0.5, min=0.1, max=5, step=0.1, description="MEC"),
    mtc=FloatSlider(value=8, min=4, max=20, step=0.5, description="MTC"),
    duration_days=IntSlider(value=7, min=3, max=14, step=1, description="Days")
);


## 13. Observation Task 3: Dose Reduction vs Interval Extension

This task compares two common dose-adjustment strategies after renal function declines, using the standard regimen under normal renal function as a reference.

### Task A: Moderate Renal Impairment

Set:

- Route = Oral
- Dose = 500 mg
- Tau = 24 h
- Vd = 70 L
- CL normal = 8 L/h
- CrCl = 30 mL/min
- $f_e$ = 0.8
- MEC = 0.5 mg/L
- MTC = 8 mg/L

Compare:

- Normal reference
- No adjustment
- Reduce dose
- Extend interval

Record and compare:

- Steady-state Cmax
- Steady-state Cmin
- AUCτ,ss
- Cavg,ss
- Time below MEC (% of dosing interval)
- Time above MTC (% of dosing interval)

Observe:

- After renal function declines, are Cavg,ss and AUC substantially higher than the normal reference when the regimen is not adjusted?
- Do both dose reduction and interval extension bring the **average steady-state exposure** closer to the normal reference?
- Does dose reduction produce a lower peak concentration and relatively smaller peak–trough fluctuation?
- Does interval extension preserve a relatively higher peak concentration but produce a lower trough and greater peak–trough fluctuation?

### Task B: Severe Renal Impairment

Change:

- CrCl = 10 mL/min

Compare the four conditions again.

Observe:

- Do Cmax, Cmin, and Cavg,ss increase further with no adjustment?
- Does Time above MTC (% of dosing interval) increase?
- Can both adjusted regimens still bring average steady-state exposure close to the normal reference?
- Why do Cmax and Cmin remain substantially different between the two adjustment strategies?

### Task C: Selecting an Adjustment Strategy

Consider:

- Why does similar average steady-state exposure not imply similar peak and trough concentrations?
- For drugs whose efficacy depends on a relatively high peak concentration, why might interval extension be more appropriate?
- For drugs that require reduced concentration fluctuation or more stable concentrations, why might dose reduction be more appropriate?
- Why is TDM often still needed to individualize therapy for drugs with a narrow therapeutic window?

> **Core concept:** The goal of renal dose adjustment is not simply to keep concentrations within a fixed range at all times. The goal is to restore appropriate drug exposure as clearance changes and to select a dose and dosing interval that match the drug’s PK/PD characteristics.


## 14. Loading Dose and Maintenance Dose

A common misconception in renal dose adjustment is:

> As long as renal function declines, all doses should be reduced by the same proportion.

In fact, loading doses and maintenance doses are determined by different factors.

### Loading Dose

A loading dose is mainly used to rapidly reach a target concentration:

$$
Loading\ Dose = \frac{Target\ Concentration \times V_d}{F}
$$

Therefore, the loading dose is mainly influenced by:

- Target concentration
- Volume of distribution, Vd
- Bioavailability, F

### Maintenance Dose

A maintenance dose is mainly used to compensate for drug clearance:

$$
Maintenance\ Dose\ Rate = \frac{Target\ Concentration \times CL}{F}
$$

Therefore, the maintenance dose is mainly influenced by:

- Target concentration
- Clearance, CL
- Bioavailability, F
- Dosing interval

A decline in renal function usually affects CL first, and therefore has a more direct effect on the maintenance dose and dosing interval.


In [ ]:
def calculate_loading_and_maintenance(
    target_concentration=5,
    vd_l=70,
    bioavailability=0.9,
    cl_l_h=8,
    tau_h=24
):
    loading_dose = target_concentration * vd_l / bioavailability
    maintenance_dose = target_concentration * cl_l_h * tau_h / bioavailability

    summary = pd.DataFrame({
        "Metric": [
            "Target concentration",
            "Vd",
            "Bioavailability",
            "CL",
            "Tau",
            "Estimated loading dose",
            "Estimated maintenance dose per interval"
        ],
        "Value": [
            f"{target_concentration:.2f} mg/L",
            f"{vd_l:.1f} L",
            f"{bioavailability:.2f}",
            f"{cl_l_h:.2f} L/h",
            f"{tau_h:.1f} h",
            f"{loading_dose:.0f} mg",
            f"{maintenance_dose:.0f} mg"
        ]
    })

    display(summary)


interact(
    calculate_loading_and_maintenance,
    target_concentration=FloatSlider(value=5, min=1, max=20, step=0.5, description="Target C"),
    vd_l=FloatSlider(value=70, min=10, max=200, step=5, description="Vd"),
    bioavailability=FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="F"),
    cl_l_h=FloatSlider(value=8, min=0.5, max=20, step=0.5, description="CL"),
    tau_h=FloatSlider(value=24, min=6, max=48, step=6, description="Tau")
);

## 15. Observation Task 5: Loading Dose and Maintenance Dose

Please complete the following tasks:

### Task A: Change Vd

Keep the other parameters unchanged and change Vd from 70 L to 140 L.

Observe:

- Does the estimated loading dose increase?
- Does the estimated maintenance dose necessarily increase by the same proportion?

### Task B: Change CL

Change Vd back to 70 L, then change CL from 8 L/h to 2 L/h.

Observe:

- Does the estimated loading dose change substantially?
- Does the estimated maintenance dose decrease?

### Task C: Clinical Interpretation

Consider:

- Why does the maintenance dose usually need to be adjusted when renal function declines?
- Why might the initial dose or loading dose still be preserved for some drugs when renal function declines?


## 16. Self-Assessment Questions: Renal Function and Dose Adjustment

Please complete the following self-assessment questions based on this notebook. It is recommended that you answer them independently before checking the reference answers in the next cell.

---

### Question 1: Which PK parameter is most directly affected by reduced renal function?

A. Emax  
B. EC50  
C. CL  
D. Hill coefficient  

---

### Question 2: What is the Cockcroft-Gault equation mainly used to estimate?

A. Hepatic clearance  
B. Creatinine clearance, CrCl  
C. Maximum drug effect, Emax  
D. Oral bioavailability, F  

---

### Question 3: For drugs that are primarily cleared by the kidneys, what is most likely to happen when CrCl decreases?

A. AUC decreases and half-life becomes shorter  
B. AUC increases and half-life becomes longer  
C. Cmax definitely becomes 0  
D. Drug effect definitely disappears completely  

---

### Question 4: Compared with extending the dosing interval, which statement about dose reduction is more reasonable?

A. The two strategies always produce exactly the same concentration curve  
B. Dose reduction usually lowers the peak concentration, while interval extension may preserve a relatively high peak concentration but produce a lower trough concentration  
C. Interval extension always increases toxicity  
D. Dose reduction always causes treatment failure  

---

### Question 5: When renal function declines, why might some drugs still retain the initial dose or loading dose?

A. Because the loading dose is mainly determined by Vd and the target concentration  
B. Because the loading dose is determined only by CL  
C. Because reduced renal function always makes Vd become 0  
D. Because no drug requires maintenance dose adjustment


## 17. Reference Answers to the Self-Assessment Questions

### Question 1

**Reference answer: C**

**Explanation:**  
Reduced renal function most directly affects the clearance, CL, of drugs that depend on renal excretion. Emax, EC50, and the Hill coefficient are PD parameters and are not the PK parameters most directly changed by reduced renal function.

---

### Question 2

**Reference answer: B**

**Explanation:**  
The Cockcroft-Gault equation is used to estimate creatinine clearance, CrCl. Many renal dose adjustment recommendations in drug prescribing information are stratified by CrCl.

---

### Question 3

**Reference answer: B**

**Explanation:**  
For drugs that are primarily cleared by the kidneys, a decrease in CrCl reduces CL. According to:

$$
AUC = \frac{Dose}{CL}
$$

and:

$$
t_{1/2} = \frac{0.693 \times V_d}{CL}
$$

A decrease in CL leads to increased AUC, prolonged half-life, and greater risk of accumulation.

---

### Question 4

**Reference answer: B**

**Explanation:**  
Both dose reduction and interval extension can lower the maintenance dose rate, but the shapes of the concentration curves are different. Dose reduction usually lowers peak concentration and fluctuation; interval extension may preserve a relatively high peak concentration, but the trough concentration at the end of the dosing interval is lower.

---

### Question 5

**Reference answer: A**

**Explanation:**  
A loading dose is mainly used to rapidly reach a target concentration and is primarily determined by the target concentration, Vd, and F. A maintenance dose is used to compensate for clearance and is mainly related to CL. Therefore, when renal function declines, the maintenance dose or dosing interval is affected more directly.


## 18. Notebook Summary

Through simulations of changes in renal function and dose adjustment, this notebook helped you understand the basic logic of individualized dosing in clinical pharmacy.

You should understand the following key points:

1. Reduced renal function mainly affects drugs that are substantially cleared by the kidneys.
2. Lower clearance increases AUC, prolongs half-life, and may increase accumulation risk.
3. The impact of renal impairment depends strongly on the fraction excreted unchanged in urine, $f_e$.
4. CrCl estimated by the Cockcroft-Gault equation is commonly used for renal dose adjustment, but it has clinical limitations.
5. Dose reduction and interval extension can both reduce exposure, but they may produce different peak and trough profiles.
6. Loading dose is mainly determined by $V_d$ and target concentration, whereas maintenance dosing is mainly determined by clearance.

The complete logic of this section can be summarized as:

$$
Renal\ function \rightarrow CL \rightarrow AUC/t_{1/2} \rightarrow Accumulation\ risk \rightarrow Dose\ adjustment
$$



## References

1. Cockcroft DW, Gault MH. Prediction of creatinine clearance from serum creatinine. *Nephron*. 1976;16(1):31-41.  
   PubMed: https://pubmed.ncbi.nlm.nih.gov/1244564/

2. U.S. Food and Drug Administration. *Pharmacokinetics in Patients with Impaired Renal Function — Study Design, Data Analysis, and Impact on Dosing*. Final guidance, March 2024.  
   FDA: https://www.fda.gov/regulatory-information/search-fda-guidance-documents/pharmacokinetics-patients-impaired-renal-function-study-design-data-analysis-and-impact-dosing
